# Task 2 - Matrix Operations (OpenMP)

6CS005 High Performance Computing. Reads pairs of matrices and computes add,
subtract, element-wise multiply, element-wise divide (NaN on divide-by-zero),
transpose of A, transpose of B, and A x B, using OpenMP.

**Runtime:** CPU is fine.

## 0. Set your repository base URL

Replace `USERNAME/REPO/BRANCH` below with your own **public** GitHub repo
so the data files download with `wget` and **anyone with the Colab link
can run this notebook end to end**. `RAW_BASE` must point at the folder
that contains `Task1 .. Task4` (the `Project` folder).

In [ ]:
RAW_BASE = "https://raw.githubusercontent.com/USERNAME/REPO/BRANCH/Project"

## 1. Check the compiler

In [ ]:
!gcc --version

## 2. Download the data from your repo

In [ ]:
!wget -q $RAW_BASE/Task2/MatData.txt
!ls -l MatData.txt

## 3. Write the source file

In [ ]:
%%writefile matrices.c
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <omp.h>

/*
 * Task 2 - Matrix operations with OpenMP (6CS005).
 *
 * Usage: ./matrices <input_file> <num_threads>
 * Output: results.txt
 *
 * The file holds matrices as a "rows,cols" header followed by that many
 * comma-separated rows (the MatData.txt / MatAdd.c format).  Matrices are
 * processed in pairs (A, B).  For each pair we do: A+B, A-B, A.*B, A./B,
 * transpose(A), transpose(B) and A*B, doing each only when the shapes allow
 * it.  Element-wise work is parallelised with collapse(2); the multiply
 * parallelises the row loop.  The number of threads is capped at the largest
 * relevant dimension.
 */

/* ---- Matrix helpers: double** with the rows/cols kept alongside. ---- */

struct Matrix {
    int rows;
    int cols;
    double **data;
};

/* Allocate a rows x cols matrix (per-row malloc, as in MatAdd.c). */
double **allocMatrix(int rows, int cols) {
    double **m = malloc(rows * sizeof(double *));
    if (m == NULL) return NULL;
    for (int r = 0; r < rows; r++) {
        m[r] = malloc(cols * sizeof(double));
        if (m[r] == NULL) {
            for (int k = 0; k < r; k++) free(m[k]);
            free(m);
            return NULL;
        }
    }
    return m;
}

/* Free every row, then the row array (MatAdd.c only freed the row array). */
void freeMatrix(double **m, int rows) {
    if (m == NULL) return;
    for (int r = 0; r < rows; r++) {
        free(m[r]);
    }
    free(m);
}

/*
 * Read one matrix from the file.
 * Returns  1 on success, 0 at clean end of file, -1 on a parse error.
 */
int readMatrix(FILE *fp, struct Matrix *out) {
    int rows, cols;
    int got = fscanf(fp, "%d,%d", &rows, &cols);
    if (got == EOF) {
        return 0;               /* no more matrices */
    }
    if (got != 2 || rows <= 0 || cols <= 0) {
        fprintf(stderr, "Invalid matrix header (got %d fields, rows=%d cols=%d)\n",
                got, rows, cols);
        return -1;
    }

    double **m = allocMatrix(rows, cols);
    if (m == NULL) {
        fprintf(stderr, "Could not allocate a %dx%d matrix\n", rows, cols);
        return -1;
    }

    for (int r = 0; r < rows; r++) {
        for (int c = 0; c < cols - 1; c++) {
            if (fscanf(fp, "%lf,", &m[r][c]) != 1) {
                fprintf(stderr, "Matrix %dx%d: bad or missing value at row %d col %d\n",
                        rows, cols, r, c);
                freeMatrix(m, rows);
                return -1;
            }
        }
        if (fscanf(fp, "%lf", &m[r][cols - 1]) != 1) {   /* last in the row */
            fprintf(stderr, "Matrix %dx%d: bad or missing value at row %d col %d\n",
                    rows, cols, r, cols - 1);
            freeMatrix(m, rows);
            return -1;
        }
    }

    out->rows = rows;
    out->cols = cols;
    out->data = m;
    return 1;
}

/* Write a result matrix to the file in the brief's format:
   a "Label - rows,cols" line, then comma-separated rows. */
void writeMatrix(FILE *out, const char *label, double **m, int rows, int cols) {
    fprintf(out, "%s - %d,%d\n", label, rows, cols);
    for (int r = 0; r < rows; r++) {
        for (int c = 0; c < cols; c++) {
            fprintf(out, "%f", m[r][c]);
            if (c < cols - 1) fprintf(out, ",");
        }
        fprintf(out, "\n");
    }
}

/* Cap the thread count at the largest dimension involved, then apply it. */
int capThreads(int requested, int maxDim) {
    int t = requested;
    if (t > maxDim) t = maxDim;
    if (t < 1) t = 1;
    omp_set_num_threads(t);
    return t;
}

int main(int argc, char **argv) {
    if (argc != 3) {
        fprintf(stderr, "Usage: %s <input_file> <num_threads>\n", argv[0]);
        return 1;
    }

    char *filename = argv[1];
    int requested = atoi(argv[2]);
    if (requested <= 0) {
        fprintf(stderr, "Number of threads must be a positive integer\n");
        return 1;
    }

    FILE *fp = fopen(filename, "r");
    if (fp == NULL) {
        fprintf(stderr, "Could not open file: %s\n", filename);
        return 1;
    }

    FILE *out = fopen("results.txt", "w");
    if (out == NULL) {
        fprintf(stderr, "Could not open results.txt for writing\n");
        fclose(fp);
        return 1;
    }

    int pair = 0;
    while (1) {
        struct Matrix A, B;

        /* Read matrix A. */
        int ra = readMatrix(fp, &A);
        if (ra == 0) break;             /* clean end of file */
        if (ra == -1) {                 /* parse error */
            fprintf(out, "Stopped: could not read matrix A of a pair.\n");
            break;
        }

        /* Read matrix B. */
        int rb = readMatrix(fp, &B);
        if (rb == 0) {                  /* odd number of matrices */
            fprintf(out, "==================== Pair %d ====================\n", pair + 1);
            fprintf(out, "Only one matrix left (%dx%d); a pair needs two. Skipped.\n",
                    A.rows, A.cols);
            freeMatrix(A.data, A.rows);
            break;
        }
        if (rb == -1) {
            fprintf(out, "Stopped: could not read matrix B of a pair.\n");
            freeMatrix(A.data, A.rows);
            break;
        }

        pair++;
        fprintf(out, "==================== Pair %d ====================\n", pair);
        fprintf(out, "Matrix A: %d,%d   Matrix B: %d,%d\n",
                A.rows, A.cols, B.rows, B.cols);

        int sameShape = (A.rows == B.rows && A.cols == B.cols);

        /* ---- Element-wise operations (need identical shapes). ---- */
        if (sameShape) {
            int t = capThreads(requested, A.rows > A.cols ? A.rows : A.cols);
            fprintf(out, "(element-wise ops using %d threads)\n", t);

            double **add = allocMatrix(A.rows, A.cols);
            double **sub = allocMatrix(A.rows, A.cols);
            double **mul = allocMatrix(A.rows, A.cols);
            double **dvd = allocMatrix(A.rows, A.cols);
            if (!add || !sub || !mul || !dvd) {
                fprintf(stderr, "Allocation failed for element-wise results\n");
                freeMatrix(add, A.rows); freeMatrix(sub, A.rows);
                freeMatrix(mul, A.rows); freeMatrix(dvd, A.rows);
                freeMatrix(A.data, A.rows); freeMatrix(B.data, B.rows);
                fclose(fp); fclose(out);
                return 1;
            }

            double start = omp_get_wtime();
            #pragma omp parallel for collapse(2)
            for (int i = 0; i < A.rows; i++) {
                for (int j = 0; j < A.cols; j++) {
                    add[i][j] = A.data[i][j] + B.data[i][j];
                    sub[i][j] = A.data[i][j] - B.data[i][j];
                    mul[i][j] = A.data[i][j] * B.data[i][j];
                    /* NaN when dividing by zero, as the brief requires. */
                    if (B.data[i][j] == 0.0) {
                        dvd[i][j] = NAN;
                    } else {
                        dvd[i][j] = A.data[i][j] / B.data[i][j];
                    }
                }
            }
            double elapsed = omp_get_wtime() - start;

            writeMatrix(out, "Addition", add, A.rows, A.cols);
            writeMatrix(out, "Subtraction", sub, A.rows, A.cols);
            writeMatrix(out, "Element-wise multiplication", mul, A.rows, A.cols);
            writeMatrix(out, "Element-wise division", dvd, A.rows, A.cols);
            printf("Pair %d: element-wise ops took %f s\n", pair, elapsed);

            freeMatrix(add, A.rows); freeMatrix(sub, A.rows);
            freeMatrix(mul, A.rows); freeMatrix(dvd, A.rows);
        } else {
            fprintf(out, "Addition cannot be done - A is %d,%d and B is %d,%d\n",
                    A.rows, A.cols, B.rows, B.cols);
            fprintf(out, "Subtraction cannot be done - different size\n");
            fprintf(out, "Element-wise multiplication cannot be done - different size\n");
            fprintf(out, "Element-wise division cannot be done - different size\n");
        }

        /* ---- Transposes (always possible). ---- */
        {
            int t = capThreads(requested, A.rows > A.cols ? A.rows : A.cols);
            double **ta = allocMatrix(A.cols, A.rows);
            if (ta != NULL) {
                #pragma omp parallel for collapse(2)
                for (int i = 0; i < A.rows; i++) {
                    for (int j = 0; j < A.cols; j++) {
                        ta[j][i] = A.data[i][j];
                    }
                }
                writeMatrix(out, "Transpose of A", ta, A.cols, A.rows);
                freeMatrix(ta, A.cols);
            }
            (void)t;
        }
        {
            int t = capThreads(requested, B.rows > B.cols ? B.rows : B.cols);
            double **tb = allocMatrix(B.cols, B.rows);
            if (tb != NULL) {
                #pragma omp parallel for collapse(2)
                for (int i = 0; i < B.rows; i++) {
                    for (int j = 0; j < B.cols; j++) {
                        tb[j][i] = B.data[i][j];
                    }
                }
                writeMatrix(out, "Transpose of B", tb, B.cols, B.rows);
                freeMatrix(tb, B.cols);
            }
            (void)t;
        }

        /* ---- Matrix multiplication A x B (needs A.cols == B.rows). ---- */
        if (A.cols == B.rows) {
            int t = capThreads(requested, A.rows);   /* row loop is parallelised */
            fprintf(out, "(matrix multiplication using %d threads)\n", t);

            double **prod = allocMatrix(A.rows, B.cols);
            if (prod != NULL) {
                int i, j, k;
                double tot;
                double start = omp_get_wtime();
                #pragma omp parallel for private(j, k, tot)
                for (i = 0; i < A.rows; i++) {
                    for (j = 0; j < B.cols; j++) {
                        tot = 0.0;
                        for (k = 0; k < A.cols; k++) {
                            tot += A.data[i][k] * B.data[k][j];
                        }
                        prod[i][j] = tot;
                    }
                }
                double elapsed = omp_get_wtime() - start;
                writeMatrix(out, "Matrix multiplication", prod, A.rows, B.cols);
                printf("Pair %d: matrix multiply took %f s\n", pair, elapsed);
                freeMatrix(prod, A.rows);
            }
        } else {
            fprintf(out, "Matrix multiplication cannot be done - "
                         "A columns (%d) do not match B rows (%d)\n",
                    A.cols, B.rows);
        }

        fprintf(out, "\n");
        freeMatrix(A.data, A.rows);
        freeMatrix(B.data, B.rows);
    }

    printf("Processed %d pair(s). Results in results.txt\n", pair);

    fclose(fp);
    fclose(out);
    return 0;
}


## 4. Compile

In [ ]:
!gcc -O2 -fopenmp -o matrices matrices.c -lm

## 5. Run

Usage: `./matrices <input_file> <num_threads>`. Output goes to `results.txt`.

In [ ]:
!./matrices MatData.txt 4

## 6. Inspect the output

In [ ]:
!head -40 results.txt

## 7. Check: 1 thread vs 8 threads give identical output

In [ ]:
!./matrices MatData.txt 1 > /dev/null && cp results.txt m1.txt
!./matrices MatData.txt 8 > /dev/null && cp results.txt m8.txt
!diff <(grep -v threads m1.txt) <(grep -v threads m8.txt) && echo 'IDENTICAL'